# N2 · 在你自己的复现上找 gap (Find Gaps in Your Own Work)

> 配套 L3(gap 分类学) + L4(idea 生成)。本专题的**灵魂**: 你不需要造练习材料 ——
> 你有 48 个真实复现, 每个都离"一篇论文"只差一个新实验。

这里把你的 `learning/reasoning-r1` (R1-Zero 复现) 当作**一篇待审稿论文**, 用 6 类 gap 雷达
扫一遍, 打分排序, 再把 top gap 翻成 idea 并过三筛。**改下面的数据成你真实的判断。**

In [1]:
import sys
from pathlib import Path
SRC = Path.cwd().parent / "src" if Path.cwd().name == "notebooks" else Path.cwd() / "src"
sys.path.insert(0, str(SRC))
import make_cards
import pandas as pd

# 指向你自己的复现专题 (存在性检查, 不存在也不报错, 只提示)
repo_root = Path.cwd()
while repo_root.name and not (repo_root / "learning").exists() and repo_root.parent != repo_root:
    repo_root = repo_root.parent
target = repo_root / "learning" / "reasoning-r1"
print("把它当待审稿论文来挖:", target, "| 存在:" , target.exists())

把它当待审稿论文来挖: C:\Workspace\dummy\learning\reasoning-r1 | 存在: True


## 6 类 gap 雷达扫描
对你的 R1-Zero 复现逐类自问。下面是**示例**判断 (基于 R1-Zero 的已知特性), 把它改成你的。
打分: Importance/Tractability/Cost 各 1–5; Priority = Imp × Tract / Cost。

In [2]:
# 每行: (类型, 一句话gap, 证据/信号, Importance, Tractability, Cost)
gaps = [
    ("⑤复现", "R1-Zero 对超参/种子极敏感, 公开方差少",
        "RL 类方法普遍单种子; 我复现时不同种子波动大", 4, 5, 1),
    ("③假设", "假设 reward(可验证答案)总是可得",
        "数学题有标准答案才好做 RL; 开放任务无此前提", 5, 3, 2),
    ("④泛化", "只在数学/代码上验证, 其它推理域未知",
        "原工作集中在可验证任务; 常识/多跳推理未测", 4, 3, 2),
    ("②评测", "只看最终答案对错, 不看推理过程是否真合理",
        "可能答案对但过程瞎蒙(回忆 process-reward 专题)", 4, 3, 2),
    ("①方法", "GRPO 去掉 critic 后优势估计噪声大",
        "组内归一化方差高, 小 batch 更明显", 3, 3, 3),
    ("⑥理论", "为什么会出现 'aha moment'(自我反思) 机理不清",
        "现象清楚, 机理无解释", 5, 2, 2),
]
df = pd.DataFrame(gaps, columns=["类型","gap","证据","Imp","Tract","Cost"])
df["Priority"] = (df["Imp"] * df["Tract"] / df["Cost"]).round(2)
df = df.sort_values("Priority", ascending=False).reset_index(drop=True)
print(df[["类型","gap","Imp","Tract","Cost","Priority"]].to_string())
print("\n→ Priority 最高的, 是'重要×可做÷便宜'最划算的起手题。")

    类型                             gap  Imp  Tract  Cost  Priority
0  ⑤复现        R1-Zero 对超参/种子极敏感, 公开方差少    4      5     1      20.0
1  ③假设            假设 reward(可验证答案)总是可得    5      3     2       7.5
2  ④泛化             只在数学/代码上验证, 其它推理域未知    4      3     2       6.0
3  ②评测           只看最终答案对错, 不看推理过程是否真合理    4      3     2       6.0
4  ⑥理论  为什么会出现 'aha moment'(自我反思) 机理不清    5      2     2       5.0
5  ①方法         GRPO 去掉 critic 后优势估计噪声大    3      3     3       3.0

→ Priority 最高的, 是'重要×可做÷便宜'最划算的起手题。


## 把 top gap 翻成 idea + 三筛
取 Priority 最高的 gap, 用 idea 五法之一翻成具体 idea, 然后过三筛 (一票否决)。

In [3]:
top = df.iloc[0]
print("最高优先级 gap:", top["类型"], "-", top["gap"], "| Priority=", top["Priority"], "\n")

idea = {
    "一句话 idea": "系统量化 R1-Zero 对随机种子的敏感性, 给出可信区间, 并提出降方差的简单改动",
    "来源 gap":    f'{top["类型"]} {top["gap"]}',
    "生成手法":    "⑤未测 + ③极限(把种子数推到统计可信)",
    "可证伪假设":  "H: GRPO 在 n≥5 种子下, 最终准确率 95% 置信区间宽度 > 3 个点(即单种子结论不可信)",
    "最小验证实验": "GPT-2-M + GSM8K 子集, 跑 5 种子, 报均值±std; 复用 reasoning-r1 现有代码; 一周出第一批点",
    "预期图":      "x=种子, y=准确率, 散点显示明显波动; 叠加'单种子会误导'的标注",
}
filters = {
    "筛1 没解决?": "RL 复现稳定性有人讨论, 但针对 R1-Zero/GRPO 的系统方差报告少 → 谨慎过(先精读相关工作)",
    "筛2 可做?":   "GPT-2-M 量级, 我 reasoning-r1 专题已有完整代码 → 强过",
    "筛3 重要?":   "so-what: '很多 R1 类结论建立在单种子上, 可能不可靠' → 会让人'哦!' → 过",
}
for k,v in idea.items():    print(f"[idea] {k}: {v}")
print()
for k,v in filters.items(): print(f"[筛] {k} {v}")

最高优先级 gap: ⑤复现 - R1-Zero 对超参/种子极敏感, 公开方差少 | Priority= 20.0 

[idea] 一句话 idea: 系统量化 R1-Zero 对随机种子的敏感性, 给出可信区间, 并提出降方差的简单改动
[idea] 来源 gap: ⑤复现 R1-Zero 对超参/种子极敏感, 公开方差少
[idea] 生成手法: ⑤未测 + ③极限(把种子数推到统计可信)
[idea] 可证伪假设: H: GRPO 在 n≥5 种子下, 最终准确率 95% 置信区间宽度 > 3 个点(即单种子结论不可信)
[idea] 最小验证实验: GPT-2-M + GSM8K 子集, 跑 5 种子, 报均值±std; 复用 reasoning-r1 现有代码; 一周出第一批点
[idea] 预期图: x=种子, y=准确率, 散点显示明显波动; 叠加'单种子会误导'的标注

[筛] 筛1 没解决? RL 复现稳定性有人讨论, 但针对 R1-Zero/GRPO 的系统方差报告少 → 谨慎过(先精读相关工作)
[筛] 筛2 可做? GPT-2-M 量级, 我 reasoning-r1 专题已有完整代码 → 强过
[筛] 筛3 重要? so-what: '很多 R1 类结论建立在单种子上, 可能不可靠' → 会让人'哦!' → 过


## 沉淀: 起 gap 卡 + idea 卡到你的个人研究库

In [4]:
my_lib = Path.cwd() / "_my_research_library"
made = []
for _ in range(len(df)):                      # 每个 gap 起一张 gap 卡
    made.append(make_cards.make_card("gap", out_dir=my_lib))
made.append(make_cards.make_card("idea", out_dir=my_lib, name="r1-seed-variance"))
print(f"已在 {my_lib.name}/ 起 {len(made)} 张卡:")
for p in made:
    print("  ", p.name)
print("\n真实使用时: 把上面 df 里每一行 gap、和这个 idea, 分别填进对应卡片。")

已在 _my_research_library/ 起 7 张卡:
   GAP-20260617-01.md
   GAP-20260617-02.md
   GAP-20260617-03.md
   GAP-20260617-04.md
   GAP-20260617-05.md
   GAP-20260617-06.md
   IDEA-20260617-01-r1-seed-variance.md

真实使用时: 把上面 df 里每一行 gap、和这个 idea, 分别填进对应卡片。


## 小结 + 下一站
你刚刚**没有读任何外部论文**, 只靠审视自己的一个复现, 就产出了 6 个 gap + 1 个过三筛的 idea。
这就是你相对其他博0 的起跑优势: 48 个现成的、你最懂其弱点的"半成品论文"。

**下一站**:
- 把这个 idea 拿去和(准)导师聊, 用导师校准 novelty/significance (→ 9.9 research-life)
- 选定后, 去 9.4 `experiment-design` 把"最小验证实验"做严谨 (baseline/消融/方差)
- 出了结果, 去 9.7 `paper-writing-submission` 写成论文 (复用 how_to_write_a_paper)

> 现在就开始: 用 `python src/make_cards.py gap --out <你的库>` 起你的第一张真卡, 持续往里填。